In [5]:
import pandas as pd
import numpy as np
import pickle
import time
from surprise import Dataset, Reader, NMF
from collections import defaultdict

# ── 1. VERİ YÜKLE ──────────────────────────────────────────
print("Veri yükleniyor...")
ratings = pd.read_csv(r'C:\Users\Huseyin\Desktop\Movie_Lens\data\rating.csv')
movies  = pd.read_csv(r'C:\Users\Huseyin\Desktop\Movie_Lens\data\movie.csv')
print(f"Toplam rating: {len(ratings):,}")

# ── 2. FİLTRELE: min_movie=50, min_user=100 ────────────────
print("\nFiltreler uygulanıyor...")
movie_counts = ratings.groupby('movieId')['rating'].count()
valid_movies = movie_counts[movie_counts >= 50].index
ratings = ratings[ratings['movieId'].isin(valid_movies)]

user_counts = ratings.groupby('userId')['rating'].count()
valid_users = user_counts[user_counts >= 100].index
ratings = ratings[ratings['userId'].isin(valid_users)]

print(f"Filtrelenmiş rating: {len(ratings):,}")
print(f"Filtrelenmiş kullanıcı: {ratings['userId'].nunique():,}")
print(f"Filtrelenmiş film: {ratings['movieId'].nunique():,}")

# ── 3. STRATİFİED SPLIT ────────────────────────────────────
print("\nStratified split yapılıyor...")

def stratified_split(df, test_size=0.2, random_state=42):
    np.random.seed(random_state)
    train_list, test_list = [], []
    for user_id, group in df.groupby('userId'):
        group = group.sample(frac=1, random_state=random_state)
        n_test = max(1, int(len(group) * test_size))
        test_list.append(group.iloc[:n_test])
        train_list.append(group.iloc[n_test:])
    return pd.concat(train_list), pd.concat(test_list)

train_df, test_df = stratified_split(ratings)
print(f"Train: {len(train_df):,}  Test: {len(test_df):,}")

# ── 4. SURPRİSE DATASET ────────────────────────────────────
reader = Reader(rating_scale=(0.5, 5.0))
train_data = Dataset.load_from_df(
    train_df[['userId','movieId','rating']], reader
).build_full_trainset()

test_data = list(zip(
    test_df['userId'],
    test_df['movieId'],
    test_df['rating']
))

# ── 5. METRİK FONKSİYONLARI ───────────────────────────────
def get_top_n(predictions, n=10):
    top_n = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        top_n[uid].append((iid, est))
    for uid, user_ratings in top_n.items():
        user_ratings.sort(key=lambda x: x[1], reverse=True)
        top_n[uid] = user_ratings[:n]
    return top_n

def compute_metrics(predictions, threshold=3.5, n=10):
    top_n = get_top_n(predictions, n)

    user_relevant = defaultdict(set)
    for uid, iid, true_r, est, _ in predictions:
        if true_r is not None and true_r >= threshold:
            user_relevant[uid].add(iid)

    precisions, ndcgs, aps = [], [], []

    for uid, user_ratings in top_n.items():
        relevant = user_relevant[uid]
        if not relevant:
            continue

        recommended = [iid for iid, _ in user_ratings]
        hits = [1 if iid in relevant else 0 for iid in recommended]

        # Precision@N
        precisions.append(sum(hits) / n)

        # NDCG@N
        dcg  = sum(h / np.log2(i+2) for i, h in enumerate(hits))
        idcg = sum(1 / np.log2(i+2) for i in range(min(len(relevant), n)))
        ndcgs.append(dcg / idcg if idcg > 0 else 0)

        # MAP
        ap, hit_count = 0, 0
        for i, iid in enumerate(recommended):
            if iid in relevant:
                hit_count += 1
                ap += hit_count / (i+1)
        aps.append(ap / min(len(relevant), n) if relevant else 0)

    return {
        'Precision@10': np.nanmean(precisions) if precisions else 0,
        'NDCG@10':      np.nanmean(ndcgs)      if ndcgs      else 0,
        'MAP':          np.nanmean(aps)         if aps        else 0,
    }

# ── 6. MODELİ EĞİT ────────────────────────────────────────
print("\n" + "="*55)
print("NMF — min_user=100, threshold=3.5, n_factors=50, n_epochs=50")
print("="*55)

algo = NMF(n_factors=50, n_epochs=50, random_state=42)

start = time.time()
algo.fit(train_data)
sure = time.time() - start
print(f"Eğitim süresi: {sure:.1f} saniye")

# ── 7. TAHMİN VE METRİKLER ────────────────────────────────
print("Tahminler hesaplanıyor...")
predictions = []
for uid, iid, true_r in test_data:
    if true_r is None:
        continue
    try:
        pred = algo.predict(uid, iid, r_ui=true_r)
        predictions.append(pred)
    except:
        pass

print(f"Toplam prediction: {len(predictions):,}")

metrics = compute_metrics(predictions, threshold=3.5, n=10)

print("\n" + "="*55)
print("SONUÇLAR")
print("="*55)
print(f"Precision@10 : {metrics['Precision@10']:.4f}")
print(f"NDCG@10      : {metrics['NDCG@10']:.4f}")
print(f"MAP          : {metrics['MAP']:.4f}")
print(f"Eğitim süresi: {sure:.1f} sn")

Veri yükleniyor...
Toplam rating: 20,000,263

Filtreler uygulanıyor...
Filtrelenmiş rating: 15,888,026
Filtrelenmiş kullanıcı: 52,481
Filtrelenmiş film: 10,524

Stratified split yapılıyor...
Train: 12,730,929  Test: 3,157,097

NMF — min_user=100, threshold=3.5, n_factors=50, n_epochs=50
Eğitim süresi: 295.3 saniye
Tahminler hesaplanıyor...
Toplam prediction: 3,157,097

SONUÇLAR
Precision@10 : 0.8102
NDCG@10      : 0.8286
MAP          : 0.7384
Eğitim süresi: 295.3 sn


In [6]:
save_path = r'C:\Users\Huseyin\Desktop\Movie_Lens\nmf_best_model_100.pkl'
with open(save_path, 'wb') as f:
    pickle.dump(algo, f)
print(f"✅ Model kaydedildi: {save_path}")

✅ Model kaydedildi: C:\Users\Huseyin\Desktop\Movie_Lens\nmf_best_model_100.pkl


In [7]:
user_id = 44230

top10 = (
    ratings[ratings['userId'] == user_id]
    .merge(movies, on='movieId')
    .sort_values('rating', ascending=False)
    .head(10)
    [['title', 'genres', 'rating']]
    .reset_index(drop=True)
)

print(f"Kullanıcı {user_id} — En Çok Beğendiği 10 Film:")
print(top10.to_string())

Kullanıcı 44230 — En Çok Beğendiği 10 Film:
                                         title                                       genres  rating
0  Seven Samurai (Shichinin no samurai) (1954)                       Action|Adventure|Drama     5.0
1                        Monsters, Inc. (2001)  Adventure|Animation|Children|Comedy|Fantasy     5.0
2                      Incredibles, The (2004)   Action|Adventure|Animation|Children|Comedy     4.5
3                Dark Knight Rises, The (2012)                  Action|Adventure|Crime|IMAX     4.5
4                      Dark Knight, The (2008)                      Action|Crime|Drama|IMAX     4.5
5                           Ratatouille (2007)                     Animation|Children|Drama     4.5
6                              Iron Man (2008)                      Action|Adventure|Sci-Fi     4.5
7                      Illusionist, The (2006)                Drama|Fantasy|Mystery|Romance     4.5
8                         Kung Fu Panda (2008)        Ac

In [8]:
user_id = 100000

top10 = (
    ratings[ratings['userId'] == user_id]
    .merge(movies, on='movieId')
    .sort_values('rating', ascending=False)
    .head(10)
    [['title', 'genres', 'rating']]
    .reset_index(drop=True)
)

print(f"Kullanıcı {user_id} — En Çok Beğendiği 10 Film:")
print(top10.to_string())

Kullanıcı 100000 — En Çok Beğendiği 10 Film:
                                    title                                           genres  rating
0                      October Sky (1999)                                            Drama     5.0
1                          Gattaca (1997)                            Drama|Sci-Fi|Thriller     5.0
2                  American Beauty (1999)                                     Comedy|Drama     5.0
3  One Flew Over the Cuckoo's Nest (1975)                                            Drama     5.0
4  Monty Python and the Holy Grail (1975)                         Adventure|Comedy|Fantasy     5.0
5                        Anastasia (1997)       Adventure|Animation|Children|Drama|Musical     5.0
6                      Stand by Me (1986)                                  Adventure|Drama     5.0
7             Beauty and the Beast (1991)  Animation|Children|Fantasy|Musical|Romance|IMAX     5.0
8               Planet of the Apes (1968)                       

In [10]:
user_counts  = ratings.groupby('userId')['rating'].count()
valid_users  = user_counts[user_counts >= 100].index
movie_counts = ratings.groupby('movieId')['rating'].count()
valid_movies = movie_counts[movie_counts >= 50].index

ratings_f = ratings[
    ratings['userId'].isin(valid_users) &
    ratings['movieId'].isin(valid_movies)
]

# İlk 5 geçerli kullanıcıyı göster
print("Geçerli kullanıcılardan ilk 5:")
print(sorted(ratings_f['userId'].unique())[:5])

Geçerli kullanıcılardan ilk 5:
[1, 3, 7, 11, 14]


In [12]:

# Filtreleme
user_counts  = ratings.groupby('userId')['rating'].count()
valid_users  = user_counts[user_counts >= 100].index
movie_counts = ratings.groupby('movieId')['rating'].count()
valid_movies = movie_counts[movie_counts >= 50].index

ratings_f = ratings[
    ratings['userId'].isin(valid_users) &
    ratings['movieId'].isin(valid_movies)
]

user_id = 1

# Kullanıcının izlediği filmler (bunlar dışındakiler aday)
rated_movie_ids = set(ratings_f[ratings_f['userId'] == user_id]['movieId'].tolist())
print(f"Kullanıcı {user_id} toplam {len(rated_movie_ids)} film izlemiş\n")

# NMF modeli yükle
with open(r'C:\Users\Huseyin\Desktop\Movie_Lens\nmf_best_model_100.pkl', 'rb') as f:
    nmf_model = pickle.load(f)

# İzlemediği tüm geçerli filmler için tahmin yap
candidate_movies = [mid for mid in valid_movies if mid not in rated_movie_ids]
print(f"Aday film sayısı: {len(candidate_movies)}")

preds = []
for mid in candidate_movies:
    try:
        est = nmf_model.predict(uid=user_id, iid=mid).est
        preds.append((mid, est))
    except:
        pass

# En iyi 30
preds.sort(key=lambda x: x[1], reverse=True)
top30 = preds[:30]

# Film adlarıyla göster
top30_df = pd.DataFrame(top30, columns=['movieId', 'tahmin'])
top30_df = top30_df.merge(movies, on='movieId')
top30_df.index = range(1, 31)

print("\nNMF — Kullanıcı 1 için Top 30 Öneri (izlemediği filmler):")
print(top30_df[['title', 'genres', 'tahmin']].to_string())

Kullanıcı 1 toplam 174 film izlemiş

Aday film sayısı: 10130

NMF — Kullanıcı 1 için Top 30 Öneri (izlemediği filmler):
                                                                                   title                                     genres    tahmin
1            Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)                                 Comedy|War  5.000000
2                                                                  Godfather, The (1972)                                Crime|Drama  5.000000
3                                                                     Rear Window (1954)                           Mystery|Thriller  5.000000
4                                          Sunset Blvd. (a.k.a. Sunset Boulevard) (1950)                    Drama|Film-Noir|Romance  5.000000
5                                                             Rashomon (Rashômon) (1950)                        Crime|Drama|Mystery  5.000000
6   Discreet Charm of the Bo